In [1]:
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import yaml
import pathlib
from pathlib import Path
import shutil

In [2]:
#--------------------------------------------------------------------------------------------------------------------------
# pdf
#--------------------------------------------------------------------------------------------------------------------------

data_name = "CT18ZNNLO-ES"
if_diag_only = False

#--------------------------------------------------------------------------------------------------------------------------
# read path
#--------------------------------------------------------------------------------------------------------------------------

before_dir = Path(f"../Predict/{data_name}")

#--------------------------------------------------------------------------------------------------------------------------
# write path
#--------------------------------------------------------------------------------------------------------------------------

after_dir = Path(f"PDF_Matrices/{data_name}")

Auxilary Functions

In [3]:
def Build_Additive_Matrix(df, prefix="error_add_"):
    cols = [col for col in df.columns if col.startswith(prefix)]
    Matrix_total = np.zeros((len(df), len(df)))
    for i in range(len(cols)):
        col = df[cols[i]]
        Matrix = np.outer(col, col)
        Matrix_total = Matrix_total + Matrix 
    return Matrix_total

DY

In [4]:
experiments =[
    'ATLAS_7',
    'ATLAS_8', 
    'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    'PHENIX',
    'STAR'
]
excludes = []

In [5]:
processes = ["DY","SIDIS"]

for process in processes:
    process_dir = after_dir / process
    if process_dir.exists():
        shutil.rmtree(process_dir) 
    process_dir.mkdir(parents=True, exist_ok=True)

In [6]:
import numpy as np
from pathlib import PurePosixPath, Path
import pickle

# 90%->68% rescale (change to 1.0 if your set is already 68% CL)
CL = 1.645
scale = 0.5 / CL

def reformat(d):
    out = {}
    for k, v in d.items():
        p = PurePosixPath(k)
        newk = PurePosixPath(*p.parts[-3:]).with_suffix("").as_posix()
        out[newk] = v
    return out

def get_roots(keys):
    # keys like "ATLAS_7/ATLAS7-00y10/3"
    return sorted({"/".join(PurePosixPath(k).parts[:2]) for k in keys})

def keys_for_root(keys, root):
    root = PurePosixPath(root)
    ks = [k for k in keys if PurePosixPath(k).parent == root]
    # sort by the trailing integer (the "number" part)
    ks.sort(key=lambda x: int(PurePosixPath(x).name))
    return ks

# Count eigen-directions from pkls 1..58 (pairs)
folder = Path(before_dir)
N_directions = int(sum(1 for _ in folder.glob("*.pkl")) / 2)
print(N_directions, "eigen-directions")

# Use member 1 to get key universe/order
with open(f"{before_dir}/1.pkl", "rb") as f:
    df_1 = reformat(pickle.load(f))
keys = list(df_1.keys())
roots = get_roots(keys)

# Load all error sets (you can also stream them, see note below)
df_plus, df_minus = {}, {}
for k in range(1, N_directions + 1):
    with open(f"{before_dir}/{2*k-1}.pkl", "rb") as f:
        df_plus[k] = reformat(pickle.load(f))
    with open(f"{before_dir}/{2*k}.pkl", "rb") as f:
        df_minus[k] = reformat(pickle.load(f))

# Build covariance matrices per root
Matrices = {}
for root in roots:
    ks = keys_for_root(keys, root)
    N_points = len(ks)

    # D has shape (K, N_points), where D[k-1, i] = Δ_i^(k)
    D = np.empty((N_directions, N_points), dtype=float)

    for k in range(1, N_directions + 1):
        plus = df_plus[k]
        minus = df_minus[k]
        # vectorize over points in this root, in a consistent order
        D[k-1, :] = [(plus[key] - minus[key]) * scale for key in ks]

    Matrices[root] = D.T @ D  # (N_points, N_points)

29 eigen-directions


In [7]:
from pathlib import Path
import pandas as pd

after_dir = Path(after_dir)

if after_dir.exists():
    shutil.rmtree(after_dir)
after_dir.mkdir(parents=True, exist_ok=True)

for file_name in (f for f in roots if f not in excludes):
    Matrix = Matrices[file_name]

    destination = after_dir / "DY" / f"{file_name}.csv"  # file_name like ATLAS_7/ATLAS7-00y10
    destination.parent.mkdir(parents=True, exist_ok=True)  # <-- creates needed folders

    if destination.exists():
        raise FileExistsError(f"{destination} already exists")

    pd.DataFrame(Matrix).to_csv(destination, index=False)
    print(f"{destination} generated")

PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_7/ATLAS7-00y10.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_7/ATLAS7-10y20.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_7/ATLAS7-20y24.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_8/ATLAS8-00y04.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_8/ATLAS8-04y08.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_8/ATLAS8-08y12.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_8/ATLAS8-116Q150.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_8/ATLAS8-12y16.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_8/ATLAS8-16y20.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_8/ATLAS8-20y24.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/ATLAS_8/ATLAS8-46Q66.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/CDF_I/CDF1.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/CDF_II/CDF2.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/CMS_13/CMS13-00y04.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/CMS_13/CMS13-04y08.csv generated
PDF_Matrices/CT18ZNNLO-ES/DY/CMS_13/CMS13-08y12.csv generated